# Homework 02: Machine Learning for Regression

**Course:** Machine Learning Zoomcamp (2026 Cohort)  
**Author:** Wisnu Anugrah Pratama  
**GitHub:** [@wapratama](https://github.com/wapratama)

---

### Overview
This notebook contains my solutions and technical write-up for **Homework 02: Machine Learning for Regression** as part of the [Machine Learning Zoomcamp](https://github.com/DataTalksClub/machine-learning-zoomcamp) by DataTalksClub.

### **Download the Dataset**
Download the official **2026 Car Fuel Efficiency** dataset directly into your `data/` directory

In [1]:
!curl -o ../data/course_lead_scoring_2026.csv https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0   0     0   0     0     0     0  --:--:-- --:--:-- --:--:--     0
 20 635180  20 130112   0     0 223245     0   0:00:02 --:--:--  0:00:02 223944
100 635180 100 635180   0     0 699771     0  --:--:-- --:--:-- --:--:-- 701081


In [2]:
import pandas as pd
import numpy as np

# Load dataset
df_raw = pd.read_csv('../data/car_fuel_efficiency_2026.csv')

# Select target and 4 feature columns
usecols = [
    'engine_displacement', 
    'horsepower', 
    'vehicle_weight', 
    'model_year', 
    'fuel_efficiency_mpg'
]

df = df_raw[usecols].copy()
df.head()

,engine_displacement,horsepower,vehicle_weight,model_year,fuel_efficiency_mpg
0,2180,243.0,3870,2006,31.9
1,2390,272.0,4210,2008,31.3
2,2320,267.0,4240,1996,27.5
3,2130,258.0,4490,1989,28.5
4,2580,304.0,4510,1994,31.0


### Homework Solution

#### Q1. Missing values
There's one column with missing values. What is it?

In [3]:
# Function to display information about missing value
def show_null(df):
    """
    Return the total missing values and the percentage of
    missing values by column.
    """
    null_count = df.isnull().sum()
    null_percentage = (null_count / df.shape[0]) * 100
    empty_count = pd.Series(((df == ' ') | (df == '')).sum())
    empty_percentage = (empty_count / df.shape[0]) * 100
    nan_count = pd.Series(((df == 'nan') | (df == 'NaN') | (df == np.nan)).sum())
    nan_percentage = (nan_count / df.shape[0]) * 100
    return pd.DataFrame({'num_missing': null_count,
                         'missing_percentage': null_percentage,
                         'num_empty': empty_count,
                         'empty_percentage': empty_percentage,
                         'nan_count': nan_count,
                         'nan_percentage': nan_percentage})

show_null(df)

,num_missing,missing_percentage,num_empty,empty_percentage,nan_count,nan_percentage
engine_displacement,0,0.00,0,0.0,0,0.0
horsepower,877,8.77,0,0.0,0,0.0
vehicle_weight,0,0.00,0,0.0,0,0.0
model_year,0,0.00,0,0.0,0,0.0
fuel_efficiency_mpg,0,0.00,0,0.0,0,0.0


The `'horsepower'` column is the **only** column with missing values.

#### Q2. Median for horsepower
What's the median (50% percentile) for variable 'horsepower'?

In [4]:
# Check the initial data descriptive statistics
df.describe()

,engine_displacement,horsepower,vehicle_weight,model_year,fuel_efficiency_mpg
count,10000.000000,9123.000000,10000.000000,10000.000000,10000.000000
mean,2268.807000,254.446016,4286.754000,1999.549500,29.997700
std,183.447355,22.844613,266.212491,14.207925,2.930245
min,1590.000000,171.000000,3320.000000,1975.000000,19.800000
25%,2150.000000,239.000000,4110.000000,1987.000000,28.000000
50%,2270.000000,254.000000,4280.000000,2000.000000,30.000000
75%,2390.000000,270.000000,4470.000000,2012.000000,31.900000
max,3110.000000,334.000000,5460.000000,2024.000000,41.200000


In [5]:
median_hp = df['horsepower'].median() 
print(f"Median horsepower: {median_hp}")

Median horsepower: 254.0


#### Prepare and split the dataset

Shuffle the filtered dataset and create the split exactly as in the lecture:

```python
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]
```

For Q5, repeat the same block with each listed seed. For Q6, use seed `9`.

In [6]:
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

# Set seed for reproducible shuffling
np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

# Split dataframes
df_train = df.iloc[idx[:n_train]].reset_index(drop=True)
df_val = df.iloc[idx[n_train:n_train + n_val]].reset_index(drop=True)
df_test = df.iloc[idx[n_train + n_val:]].reset_index(drop=True)

# Extract target vectors
y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values
y_test = df_test['fuel_efficiency_mpg'].values

# Remove target from feature matrices to prevent data leakage
del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

In [13]:
display(df_train.head())
df_train.info()

,engine_displacement,horsepower,vehicle_weight,model_year
0,1900,239.0,3790,2015
1,2000,224.0,4380,1992
2,2330,286.0,4090,2022
3,2300,NaN,4150,1997
4,2340,269.0,4400,2001


<class 'pandas.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   engine_displacement  6000 non-null   int64  
 1   horsepower           5462 non-null   float64
 2   vehicle_weight       6000 non-null   int64  
 3   model_year           6000 non-null   int64  
dtypes: float64(1), int64(3)
memory usage: 187.6 KB


In [15]:
display(df_val.head())
df_val.info()

,engine_displacement,horsepower,vehicle_weight,model_year
0,2240,265.0,3990,2004
1,2390,259.0,4770,2019
2,2360,265.0,4320,2018
3,2430,279.0,4430,2008
4,2220,249.0,4030,1999


<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   engine_displacement  2000 non-null   int64  
 1   horsepower           1824 non-null   float64
 2   vehicle_weight       2000 non-null   int64  
 3   model_year           2000 non-null   int64  
dtypes: float64(1), int64(3)
memory usage: 62.6 KB


In [16]:
display(df_test.head())
df_test.info()

,engine_displacement,horsepower,vehicle_weight,model_year
0,2370,268.0,4410,1981
1,2240,256.0,4220,2020
2,2090,217.0,3910,2023
3,2280,251.0,4420,2021
4,2450,256.0,4420,1979


<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   engine_displacement  2000 non-null   int64  
 1   horsepower           1837 non-null   float64
 2   vehicle_weight       2000 non-null   int64  
 3   model_year           2000 non-null   int64  
dtypes: float64(1), int64(3)
memory usage: 62.6 KB


#### Q3. Filling NAs
* We need to deal with missing values for the column from Q1.
* We have two options: fill it with 0 or with the mean of this variable.
* Try both options. For each, train a linear regression model without regularization using the code from the lessons.
* For computing the mean, use the training only!
* Use the validation dataset to evaluate the models and compare the RMSE of each option.
* Round the RMSE scores to 3 decimal digits using `round(score, 3)`. This
  keeps the imputation difference visible in this release.
* Which option gives better RMSE?

Options:

- With 0
- With mean
- Both are equally good

In [24]:
# Define helper functions for RMSE and the Normal Equation
def rmse(y_actual, y_pred):
    """Computes Root Mean Squared Error between actual and predicted values."""
    error = y_actual - y_pred
    mse = (error ** 2).mean()
    return np.sqrt(mse)

def train_linear_regression(X, y):
    """Trains unregularized linear regression using the Normal Equation."""
    ones = np.ones(X.shape[0]) 
    
    X = np.column_stack([ones, X])  # Add the single bias column
    
    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)
    
    # Return the scalar bias (w_full[0]) and the rest as feature weights
    return w_full[0], w_full[1:]

In [25]:
# Option A: Fill missing values with 0
X_train_zero = df_train.fillna(0).values
w0_zero, w_zero = train_linear_regression(X_train_zero, y_train)

X_val_zero = df_val.fillna(0).values
y_pred_zero = w0_zero + X_val_zero.dot(w_zero)
rmse_zero = rmse(y_val, y_pred_zero)

print(f"RMSE with 0: {round(rmse_zero, 3)}")

RMSE with 0: 2.205


In [26]:
# Option B: Fill missing values with mean of training set ONLY
hp_mean = df_train['horsepower'].mean()

df_train_mean = df_train.copy()
df_train_mean['horsepower'] = df_train_mean['horsepower'].fillna(hp_mean)
X_train_mean = df_train_mean.values

w0_mean, w_mean = train_linear_regression(X_train_mean, y_train)

df_val_mean = df_val.copy()
df_val_mean['horsepower'] = df_val_mean['horsepower'].fillna(hp_mean)
X_val_mean = df_val_mean.values

y_pred_mean = w0_mean + X_val_mean.dot(w_mean)
rmse_mean = rmse(y_val, y_pred_mean)

print(f"RMSE with mean: {round(rmse_mean, 3)}")

RMSE with mean: 2.202


The correct choice is **With mean**. Since `2.202` is lower than `2.205` by `0.003`, filling missing values with the **mean** produces slightly better predictions on the validation set than filling them with `0`

#### Q4. Best regularization 
* Now let's train a regularized linear regression.
* For this question, fill the NAs with 0. 
* Try different values of `r` from this list: `[0, 0.01, 0.1, 1, 5, 10, 100]`.
* Use RMSE to evaluate the model on the validation dataset.
* Round the RMSE scores to 4 decimal digits. This keeps the small but real
  regularization differences visible instead of turning several choices into a tie.
* Which `r` gives the best RMSE?

If multiple options give the same best RMSE, select the smallest `r`.

Options:

- 0
- 0.01
- 0.1
- 1
- 5
- 10
- 100

In [30]:
# Define helper functions for the Normal Equation with Regularization
def train_linear_regression_reg(X, y, r=0.001):
    """Trains regularized linear regression (Ridge)."""
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    
    XTX = X.T.dot(X)
    reg_matrix = r * np.eye(XTX.shape[0])
    XTX = XTX + reg_matrix
    
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)
    
    return w_full[0], w_full[1:]

In [31]:
# Evaluate r across candidate list
r_candidates = [0, 0.01, 0.1, 1, 5, 10, 100]
X_train_0 = df_train.fillna(0).values
X_val_0 = df_val.fillna(0).values

for r in r_candidates:
    w0_r, w_r = train_linear_regression_reg(X_train_0, y_train, r=r)
    y_pred_r = w0_r + X_val_0.dot(w_r)
    score = rmse(y_val, y_pred_r)
    print(f"r = {r:6}: RMSE = {round(score, 4)}")

r =      0: RMSE = 2.2053
r =   0.01: RMSE = 2.2058
r =    0.1: RMSE = 2.2241
r =      1: RMSE = 2.3492
r =      5: RMSE = 2.4094
r =     10: RMSE = 2.4195
r =    100: RMSE = 2.4292


The correct answer for Question 4 is 0. Because lower RMSE is better. The goal when tuning the regularization parameter $r$ is to find the value that yields the lowest error on the validation set.

#### Q5. RMSE standard deviation
* We used seed 42 for splitting the data. Let's find out how selecting the seed influences our score.
* Try different seed values: `[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]`.
* For each seed, do the train/validation/test split with 60%/20%/20% distribution.
* Fill the missing values with 0 and train a model without regularization.
* For each seed, evaluate the model on the validation dataset and collect the RMSE scores. 
* What's the standard deviation of all the scores? To compute the standard deviation, use `np.std`.
* Round the result to 3 decimal digits (`round(std, 3)`)

What's the value of std?

> Note: Standard deviation shows how different the values are.
> If it's low, then all values are approximately the same.
> If it's high, the values are different. 
> If standard deviation of scores is low, then our model is *stable*.

In [32]:
seeds = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
scores = []

for s in seeds:
    np.random.seed(s)
    idx_s = np.arange(n)
    np.random.shuffle(idx_s)
    
    df_tr = df.iloc[idx_s[:n_train]].reset_index(drop=True)
    df_v = df.iloc[idx_s[n_train:n_train + n_val]].reset_index(drop=True)
    
    y_tr = df_tr['fuel_efficiency_mpg'].values
    y_v = df_v['fuel_efficiency_mpg'].values
    
    del df_tr['fuel_efficiency_mpg']
    del df_v['fuel_efficiency_mpg']
    
    X_tr = df_tr.fillna(0).values
    X_v = df_v.fillna(0).values
    
    w0_s, w_s = train_linear_regression(X_tr, y_tr)
    y_pred_s = w0_s + X_v.dot(w_s)
    scores.append(rmse(y_v, y_pred_s))

std_score = np.std(scores)
print(f"Standard deviation of RMSEs: {round(std_score, 3)}")

Standard deviation of RMSEs: 0.029


#### Q6. Evaluation on test
* Split the dataset like previously, use seed 9.
* Combine train and validation datasets.
* Fill the missing values with 0 and train a model with `r=0.001`. 

What's the RMSE on the test dataset?

In [33]:
# Train/Val/Test split with seed 9
np.random.seed(9)
idx_9 = np.arange(n)
np.random.shuffle(idx_9)

df_tr9 = df.iloc[idx_9[:n_train]].reset_index(drop=True)
df_v9 = df.iloc[idx_9[n_train:n_train + n_val]].reset_index(drop=True)
df_te9 = df.iloc[idx_9[n_train + n_val:]].reset_index(drop=True)

y_tr9 = df_tr9['fuel_efficiency_mpg'].values
y_v9 = df_v9['fuel_efficiency_mpg'].values
y_te9 = df_te9['fuel_efficiency_mpg'].values

del df_tr9['fuel_efficiency_mpg']
del df_v9['fuel_efficiency_mpg']
del df_te9['fuel_efficiency_mpg']

In [34]:
# Combine Train + Validation sets
df_full_train = pd.concat([df_tr9, df_v9]).reset_index(drop=True)
y_full_train = np.concatenate([y_tr9, y_v9])

# Train on combined data with r = 0.001
X_full_train = df_full_train.fillna(0).values
X_test = df_te9.fillna(0).values

w0_final, w_final = train_linear_regression_reg(X_full_train, y_full_train, r=0.001)

# Evaluate on Test set
y_pred_test = w0_final + X_test.dot(w_final)
test_rmse = rmse(y_te9, y_pred_test)

print(f"Final Test RMSE: {round(test_rmse, 3)}")

Final Test RMSE: 2.236
